In [29]:
#Imports
from langchain_openai import OpenAIEmbeddings
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [8]:
#Initialization of topic and user_profile variable.
topic = {
    "array": 0,
    "dictionary" : 0,
    "linked list": 0,
    "tree": 0,
    "graph": 0
}


user_profile = {
    "topic_strength": topic,
    "learning_style": None
}



In [9]:
#main - chatbased CLI
print("Welcome! Let's personalize your CS study notes!Please answer the following questions.")

#Initialization of questions
initial_topic_questions =["How comfortable are you with Arrays?",
                     "How comfortable are you with Dictionary?",
                     "How comfortable are you with Linked List?",
                     "How comfortable are you with Tree?",
                     "How comfortable are you with Graph?"]
initial_topic_answers = []

initial_personalization_questions = [
    "Do you prefer direct steps or detailed discussion? Press 1 for direct steps, 2 for detailed discussion",
    "Would you prefer a checklist or conversation? Press 1 for checklist, 2 for conversation"
]
initial_personalization_answers = []
#getting user's input for the initial topic based questions

for i in range(len(initial_topic_questions)):
    print(initial_topic_questions[i], flush = True)
    initial_topic_answers.append(input("Give answer between 1-5"))

for j in range(len(initial_personalization_questions)):
    print(initial_personalization_questions[j],flush = True)
    initial_personalization_answers.append(input())
    

Welcome! Let's personalize your CS study notes!Please answer the following questions.
How comfortable are you with Arrays?
How comfortable are you with Dictionary?
How comfortable are you with Linked List?
How comfortable are you with Tree?
How comfortable are you with Graph?
Do you prefer direct steps or detailed discussion? Press 1 for direct steps, 2 for detailed discussion
Would you prefer a checklist or conversation? Press 1 for checklist, 2 for conversation


In [10]:
#assign initial topic answers to topic dictionary

for key, value in zip(topic, initial_topic_answers):
    topic[key] = int(value)

print(topic)

#determining learning style
for k in range(len(initial_personalization_answers)-1):
    if initial_personalization_answers[k] == "1" and initial_personalization_answers[k+1] == "1":
        user_profile["learning_style"] = "action-based"
    elif initial_personalization_answers[k] == "2" and initial_personalization_answers[k+1] == "2":
        user_profile["learning_style"] ="relationship-based"
    else:
        user_profile["learning_style"] = "mixed"

print(user_profile)

{'array': 1, 'dictionary': 2, 'linked list': 3, 'tree': 4, 'graph': 5}
{'topic_strength': {'array': 1, 'dictionary': 2, 'linked list': 3, 'tree': 4, 'graph': 5}, 'learning_style': 'action-based'}


In [11]:
#Generate notes based on weak topics

def find_weakest_topic(user_profile):
    min_key = min(user_profile["topic_strength"], key =user_profile["topic_strength"].get)

    return min_key
print(find_weakest_topic(user_profile))
    

array


In [12]:
from dotenv import load_dotenv
import os
from openai import OpenAI

# Load environment variables from .env file
load_dotenv()

# Access the API key 
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

In [14]:
#Generates notes based on user's learning style and weak topic
full_prompt= f""""
You are an expert Python-based study note generator for computer science students.
Generate concise and helpful study notes on the user's weakest topic: {find_weakest_topic(user_profile)}.
Adapt the tone and structure based on the user's learning style: {user_profile["learning_style"]}.

Use the following formatting rules based on the learning style:

If learning_style is "action-based", structure the output as:

- Simple, direct sentences
- Concise bullet points
- Task- and outcome-focused content
- Give structure coding example

If learning_style is "relationship-based", structure the output as:

- Simple, direct sentences
- A narrative or dialogue-style explanation
- Paragraph format with emotional/contextual cues to build understanding
- Give structure coding example

Ensure the content remains clear, engaging, and easy to follow regardless of the style.
"""


response = client.chat.completions.create(
        model="gpt-4o-mini",  # or "gpt-3.5-turbo", or "gpt-4o-mini"
        messages=[
            {"role": "user",
            "content": full_prompt},
        ]
    )

    # Print the response text
response_text = response.choices[0].message.content
print(response_text)

### Study Notes on Arrays (Action-Based)

- **Understand the Concept**: An array is a collection of items stored at contiguous memory locations.
  
- **Key Operations**:
  - **Create**: Define an array with fixed size.
  - **Access**: Retrieve an element using its index.
  - **Update**: Change the value of an element at a specific index.
  - **Delete**: Remove an element and potentially shift others.

- **Array Indexing**:
  - Starts at 0 in Python.
  - Last index is `length of array - 1`.

- **Common Methods**:
  - Use `append()` to add an element.
  - Use `remove()` to delete an element.

- **Example Code**:

```python
# Creating an array
my_array = [1, 2, 3, 4, 5]

# Accessing an element
print(my_array[0])  # Output: 1

# Updating an element
my_array[1] = 10
print(my_array)  # Output: [1, 10, 3, 4, 5]

# Appending an element
my_array.append(6)
print(my_array)  # Output: [1, 10, 3, 4, 5, 6]

# Removing an element
my_array.remove(10)
print(my_array)  # Output: [1, 3, 4, 5, 6]
```

- *

In [13]:
#Note feedback loop

survey =[
    {
        "id": 1,
        "question": "How helpful was this note?",
        "options": ["Very helpful and clear", "Somewhat helpful and could be clearer", "Not helpful"]
    },
    {
        "id": 2,
        "question": "How would you describe this note?",
        "options": ["Straight forward and to the point", "Detailed and storylike"]
    },
    {
        "id": 3,
        "question": "What would you prefer more in this note?",
        "options": ["More step by step instructions", "More background, context, stories"]
    },
    {
        "id": 4,
        "question": "Overall does this note match your learning style?",
        "options": ["Perfect match", "Needs more details, stories, example", "Needs more concise example"]
    }
]

In [9]:
#Note survey function
response = []
for i in range(len(survey)):
    print(survey[i]["question"])
    for j in range (len(survey[i]["options"])):
        print(f"{j+1}. {survey[i]["options"][j]}")
    response.append(input("Press only 1 desired number"))


How helpful was this note?
1. Very helpful and clear
2. Somewhat helpful and could be clearer
3. Not helpful
How would you describe this note?
1. Straight forward and to the point
2. Detailed and storylike
What would you prefer more in this note?
1. More step by step instructions
2. More background, context, stories
Overall does this note match your learning style?
1. Perfect match
2. Needs more details, stories, example
3. Needs more concise example


In [ ]:
#Add code to incorporate these survey results in this note

In [ ]:
from llama_index.core.node_parser import (
    SemanticSplitterNodeParser,
)
from llama_index.embeddings.openai import OpenAIEmbedding

embed_model = OpenAIEmbedding()
splitter = SemanticSplitterNodeParser(
    buffer_size=5, breakpoint_percentile_threshold=30, embed_model=embed_model
)

In [16]:
from llama_index.core import Document
doc = Document(text=response_text)

In [17]:
nodes = splitter.get_nodes_from_documents([doc]) 

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=500,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

In [20]:
texts = text_splitter.create_documents([response_text])

In [23]:
print(texts[2])
len(texts)

page_content='# Removing an element
my_array.remove(10)
print(my_array)  # Output: [1, 3, 4, 5, 6]
```

- **Remember**: Practice the above operations to gain confidence in using arrays. Focus on manipulating data for practical tasks. 

- **Next Steps**: Implement a small project using arrays to solidify your understanding—like a simple task manager or a score tracker for games!'


3

In [31]:
#add faiss to the chunks

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [32]:
#get whats the format for texts

library = FAISS.from_documents(texts, embeddings)

In [33]:
#save faiss index

library.save_local("faiss_index_relationship_chunks")

In [34]:
r_chunk_saved = FAISS.load_local("faiss_index_relationship_chunks", embeddings, allow_dangerous_deserialization=True)

In [35]:


for faiss_id in range(r_chunk_saved.index.ntotal):
    # Get the docstore ID
    docstore_id = r_chunk_saved.index_to_docstore_id[faiss_id]
    doc = r_chunk_saved.docstore._dict[docstore_id]
    chunk_text = doc.page_content

    # Prompt for quiz generation
    prompt = f"""
    Based on the following text chunk, generate 2-5 multiple choice questions
    with answers and explanations. Minimum 2 questions, maximum 5 based on relavance. 
    
    - Do not focus on the analogy. Make questions based on programming concept present in the chunk.
    - If you're asking coding based question, make sure to give the relavant code before asking. For example, if you ask what is the output of fruits[0] then surely give fruits array. 
    - Don't make the answer choices obvious. Focus on programming concepts.  
    
    For each question, provide the correct answer immediately after the question, labeled with "ANSWER:", and then provide the explanation labeled with "EXPLANATION:".

    Example format:
    1. What is the output of the following code?
    A) Option A
    B) Option B
    C) Option C
    D) Option D

    ANSWER: B

    EXPLANATION: This is why B is the correct answer.

    Text chunk:
    {chunk_text}
    """

    # Call OpenAI
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a Python language based Computer Science quiz generator."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )

    # Print the result
    response_text += f"\n\nChunk {faiss_id}\n\n"
    response_text += response.choices[0].message.content
print(response_text)

### Study Notes on Arrays (Action-Based)

- **Understand the Concept**: An array is a collection of items stored at contiguous memory locations.
  
- **Key Operations**:
  - **Create**: Define an array with fixed size.
  - **Access**: Retrieve an element using its index.
  - **Update**: Change the value of an element at a specific index.
  - **Delete**: Remove an element and potentially shift others.

- **Array Indexing**:
  - Starts at 0 in Python.
  - Last index is `length of array - 1`.

- **Common Methods**:
  - Use `append()` to add an element.
  - Use `remove()` to delete an element.

- **Example Code**:

```python
# Creating an array
my_array = [1, 2, 3, 4, 5]

# Accessing an element
print(my_array[0])  # Output: 1

# Updating an element
my_array[1] = 10
print(my_array)  # Output: [1, 10, 3, 4, 5]

# Appending an element
my_array.append(6)
print(my_array)  # Output: [1, 10, 3, 4, 5, 6]

# Removing an element
my_array.remove(10)
print(my_array)  # Output: [1, 3, 4, 5, 6]
```

- *

In [36]:
#formatting AI generated questions into chunk based questions
import re
inner_dictionary = {
    "question": None,
    "options": [],
    "answer": None,
    "explanation": None 
}

outer_dictionary = {} #add chunk numebr here 

for lines in response_text.splitlines():
    line = lines.strip()
    
    if line.startswith("Chunk"):
        current_chunk_id = line
        outer_dictionary[current_chunk_id] = {}
        continue
        
    match = re.match(r'^\d+\.', line)
    
    if match:
        question_number = match.group(0)[:-1]
        
        inner_dictionary= {
            "question": line.replace(match.group(0), "").strip(),
            "options": [],
            "answer": None,
            "explanation": None,
            "user_answer": None
        }
        #FIX code is not there
        if current_chunk_id:
            outer_dictionary[current_chunk_id][question_number] = inner_dictionary
            current_question_id = question_number
        continue
        
    if line.startswith(("A)", "B)", "C)", "D)", "E)")):
        if current_chunk_id and current_question_id:
            outer_dictionary[current_chunk_id][current_question_id]["options"].append(line)
        continue
        
    if line.startswith("ANSWER:"):
        if current_chunk_id and current_question_id:
            answer_text = line.split(":", 1)[1].strip()
            outer_dictionary[current_chunk_id][current_question_id]["answer"] = answer_text
        continue
        
    if line.startswith("EXPLANATION:"):
        if current_chunk_id and current_question_id:
            explanation_text = line.split(":", 1)[1].strip()
            outer_dictionary[current_chunk_id][current_question_id]["explanation"] = explanation_text
        continue
        
        
print(outer_dictionary)
print(inner_dictionary)




{'Chunk 0': {'1': {'question': 'What will be the output of the following code?', 'options': ['A) 30', 'B) 40', 'C) 50', 'D) IndexError'], 'answer': 'B', 'explanation': 'In Python, array indexing starts at 0. Therefore, `numbers[3]` accesses the fourth element in the list, which is 40. The correct output is B.', 'user_answer': None}, '2': {'question': 'Given the code below, what will be the result after executing the update operation?', 'options': ["A) ['apple', 'banana', 'cherry']", "B) ['apple', 'orange', 'cherry']", "C) ['orange', 'banana', 'cherry']", "D) ['apple', 'banana']"], 'answer': 'B', 'explanation': "The code updates the element at index 1 of the `fruits` list from 'banana' to 'orange'. After the update, the list becomes ['apple', 'orange', 'cherry']. Thus, the correct answer is B.", 'user_answer': None}, '3': {'question': 'What will happen if you try to access the following index in the code below?', 'options': ['A) red', 'B) green', 'C) blue', 'D) IndexError'], 'answer': '

In [ ]:
# add code to take user's input as answer for quiz questions
#show quiz question and options first
#take user input
#check correct or wrong
#show explanation and correct answer
#think how to use this answers later for bayesian layer

# think a way - how can you utilize this correct answer?  
#if wrong, we add the question in another wrong question bank with chunk number and feed it to LLM to strength on that topic
from collections import defaultdict

correct_count = 0
wrong_count = 0
wrong_question_bank = defaultdict(list)

for outer_key,outer_value in outer_dictionary.items():
    for inner_key,inner_value in outer_value.items():
        print(f"""{inner_key}.{inner_value["question"]}""", flush=True)
        for i in inner_value["options"]:
            print(i)
        inner_value["user_answer"] = input("My answer: ")
        if inner_value["answer"] == inner_value["user_answer"]:
            correct_count += 1
        else:
            #We have question and chunk number associated with it here.
            wrong_question_bank[outer_key].append(inner_value["question"])
            wrong_count += 1
                
print(wrong_question_bank)

1.What will be the output of the following code?
A) 30
B) 40
C) 50
D) IndexError
2.Given the code below, what will be the result after executing the update operation?
A) ['apple', 'banana', 'cherry']
B) ['apple', 'orange', 'cherry']
C) ['orange', 'banana', 'cherry']
D) ['apple', 'banana']
3.What will happen if you try to access the following index in the code below?
A) red
B) green
C) blue
D) IndexError
4.What is the result of the following code snippet?
A) [1, 2, 3, 4, 5]
B) [1, 2, 4, 5]
C) [1, 2, 3, 5]
D) [2, 3, 4, 5]
1.What will be the output of the following code after executing all lines?
A) [1, 2, 3, 4, 5, 6]
B) [1, 10, 3, 4, 5]
C) [1, 10, 3, 4, 5, 6]
D) [10, 2, 3, 4, 5, 6]
2.If you wanted to remove the element '10' from `my_array` after executing the provided code, which method would you use?
A) delete(10)
B) discard(10)
C) remove(10)
D) pop(10)
3.After executing the following code, what will be the value of `my_array[1]`?
A) 2
B) 10
C) 1
D) 3
4.What does the `append()` method d

In [38]:
for value in wrong_question_bank.values():
    print(value)

['What will be the output of the following code?', 'Given the code below, what will be the result after executing the update operation?', 'What will happen if you try to access the following index in the code below?', 'What is the result of the following code snippet?']
['What will be the output of the following code after executing all lines?', "If you wanted to remove the element '10' from `my_array` after executing the provided code, which method would you use?", 'After executing the following code, what will be the value of `my_array[1]`?', 'What does the `append()` method do in the context of the provided code?']
['Given the following code, what will happen if the element `10` is not present in the array when the `remove()` method is called?', 'What is the purpose of the `remove()` method in the context of list manipulation?', 'If you wanted to remove all occurrences of a value from a list, which method would be more appropriate than `remove()`?']


In [39]:
#calculate bayesian eq
# n = number of question

def bayes_mastery_posterior(n=wrong_count+correct_count, k=correct_count, prior=0.5, p_correct_mastered=0.9, p_correct_not_mastered=0.25):
 
    p_data_given_M = (p_correct_mastered ** k) * ((1 - p_correct_mastered) ** (n - k))
    p_data_given_notM = (p_correct_not_mastered ** k) * ((1 - p_correct_not_mastered) ** (n - k))
    
    # Bayes rule
    numerator = p_data_given_M * prior
    denominator = numerator + p_data_given_notM * (1 - prior)
    
    posterior = numerator / denominator if denominator > 0 else 0
    return posterior

In [40]:
posterior = bayes_mastery_posterior()
print(posterior)

8.523708776964664e-10


In [ ]:
posterior = bayes_mastery_posterior()

full_prompt= f""""
You are an expert Python-based study note generator for computer science students.
Student took a MCQ test on topic {find_weakest_topic(user_profile)} and performed {posterior}.
If posterior is <0.3 it means user doesn't know anything about the topic so give easy to read and clear, concise notes again. 
If posterior is ~0.6 it means user partially knows about the topic so review the wrong questions {wrong_question_bank} and focus on those topics.
If posterior is >0.9 it means user knows about the topic so proceed to advanced learning of this topic.

Also, based on posterior discuss which questions and sub topics user got wrong. If you are discussing specific questions from programming questions, must mention code associated with the question.

Adapt the tone and structure based on the user's learning style: {user_profile["learning_style"]}.

Use the following formatting rules based on the learning style:

If learning_style is "action-based", structure the output as:

- Simple, direct sentences
- Concise bullet points
- Task- and outcome-focused content
- Give structure coding example

If learning_style is "relationship-based", structure the output as:

- Simple, direct sentences
- A narrative or dialogue-style explanation
- Paragraph format with emotional/contextual cues to build understanding
- Give structure coding example

Ensure the content remains clear, engaging, and easy to follow regardless of the style.
"""


response = client.chat.completions.create(
        model="gpt-4o-mini",  # or "gpt-3.5-turbo", or "gpt-4o-mini"
        messages=[
            {"role": "user",
            "content": full_prompt},
        ]
    )

    # Print the response text
response_text = response.choices[0].message.content
print(response_text)

Given your performance on the topic of arrays, we'll review the necessary concepts and address the questions you've struggled with. Here’s a structured approach to enhance your understanding. 

### Topic: Arrays in Python

#### Key Concepts to Review:

- **Array Definition**: 
  - An array is a data structure that holds a fixed-size sequential collection of elements of the same type.

- **Common Operations**:
  - **Accessing Elements**: 
    - Use an index: `my_array[index]`
  - **Updating Elements**: 
    - Assign a new value to an index: `my_array[index] = new_value`
  - **Adding Elements**:
    - **Append**: `my_array.append(value)` - adds to the end of the array.
    - **Insert**: `my_array.insert(index, value)` - adds at specified position.
  - **Removing Elements**:
    - **Remove**: `my_array.remove(value)` - removes the first occurrence of a value.
    - **Pop**: `my_array.pop(index)` - removes and returns element at specified index.

#### Specific Questions to Focus On:

You p